# Player Level Progression — Simple

Minimal view of the **full population** (no A/B split) over a **90-day install lookback**:
1. Average + percentile (P10/P50/P90) max level reached by days since install
2. Cumulative level-reach funnel

All cohorts included have matured at least `days_since_install_cap` days, and the days-since-install axis is capped at that same value, so every cohort is observed over an equal, comparable window.

In [114]:
# hide-output
# Import libraries and initialise the BigQuery connector
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html

bqc = BigQueryConnector()

## Get data

### Setting parameters

In [115]:
# Complete population, no A/B windowing — but cohorts must be comparable:
# every included cohort has had at least `days_since_install_cap` days to mature,
# and no cohort is looked at beyond that many days since install.
import datetime as dt

days_since_install_cap = 28  # parameter: max days since install considered in every chart
install_lookback_days = 90   # parameter: how many days of install cohorts to include

end_date = dt.datetime.today() - dt.timedelta(days=1)                          # latest activity date available
install_cutoff_date = end_date - dt.timedelta(days=days_since_install_cap)      # newest cohort allowed — must have matured `days_since_install_cap` days
start_date = install_cutoff_date - dt.timedelta(days=install_lookback_days)     # oldest cohort included

print(f"Install window: {start_date.strftime('%Y-%m-%d')} to {install_cutoff_date.strftime('%Y-%m-%d')}")
print(f"Activity data through: {end_date.strftime('%Y-%m-%d')}")
print(f"Days since install cap: {days_since_install_cap}")

Install window: 2026-04-01 to 2026-06-30
Activity data through: 2026-07-28
Days since install cap: 28


In [116]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = False

### Max level by day

In [117]:
# hide-output
# Estimate query cost for player level SQL before executing
query_location = './sql/playerlevel_last90.sql'
parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'install_cutoff_date': install_cutoff_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'),
    'exclude_networks': ['CPE'],
}

cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 35.42 GB when run.
Estimated query cost: $0.24


In [118]:
# hide-output
# Fetch player level data from BigQuery or load from local pickle cache
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playerlevel_last90.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playerlevel_last90.pkl')

In [119]:
# hide-output
# Preview raw player level data
data

,user_id,dt,install_dt,country_code,days_since_install,max_level,platform,acquisition_type,install_build_version
0,396FEC7AADDE50F4,2026-04-01,2026-04-01,BR,0,6,AND,Non-Attributed,0.72.1
1,2CA6F94C5F7359FF,2026-05-29,2026-04-01,BR,58,21,AND,Non-Attributed,0.72.1
2,456B61ED271635FC,2026-06-11,2026-04-01,AR,71,41,AND,Non-Attributed,0.72.1
3,75884781C5464159,2026-04-03,2026-04-01,BR,2,5,AND,Non-Attributed,0.72.1
4,A1665AA88EC78397,2026-04-08,2026-04-01,ES,7,10,IOS,Non-Attributed,0.72.1
...,...,...,...,...,...,...,...,...,...
471691,18B42136D8F4DCF6,2026-06-30,2026-06-30,US,0,2,IOS,Non-Attributed,0.78.0
471692,49FA94331B9C272E,2026-07-03,2026-06-30,US,3,8,AND,Non-Attributed,0.77.0
471693,5FEDA941B224DB20,2026-07-21,2026-06-30,SE,21,21,AND,Non-Attributed,0.78.0
471694,E0C8CADE640DC720,2026-07-23,2026-06-30,US,23,30,AND,Influencer,0.78.0


### Max level per hour

In [120]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
minute_refresh_data = False

In [121]:
# hide-output
# Estimate query cost for the session-level Day 0 minute SQL before executing
minutes_since_install_cap = 1440  # parameter: max minutes since install considered — 1440 = Day 0 (first 24h)

minute_query_location = './sql/playerlevel_day0_minute.sql'
minute_parameters = {
    'start_date': start_date.strftime('%Y-%m-%d'),
    'install_cutoff_date': install_cutoff_date.strftime('%Y-%m-%d'),
    'end_date': end_date.strftime('%Y-%m-%d'),
    'minutes_cap': minutes_since_install_cap,
    'exclude_networks': ['CPE'],
}

cost_info = bqc.print_cost_estimate(query=minute_query_location, is_path=True, query_parameters=minute_parameters)

This query will process 10.34 GB when run.
Estimated query cost: $0.07


In [122]:
# hide-output
# Fetch session-level Day 0 minute data from BigQuery or load from local pickle cache
minute_data = pd.DataFrame()

if minute_refresh_data:
    minute_data = bqc.get(query=minute_query_location, is_path=True, query_parameters=minute_parameters)
    minute_data.to_pickle('./data/playerlevel_day0_minute.pkl')
else:
    # Load from local cache to avoid repeated query costs
    minute_data = pd.read_pickle('./data/playerlevel_day0_minute.pkl')

In [123]:
# hide-output
# Preview raw session-level Day 0 data
minute_data

,user_id,session_id,session_end_ts,install_ts,minutes_since_install,max_level,platform,acquisition_type,install_build_version
0,6F5CCD2D33C05371,fca66a0e-013b-4f1c-bfd2-14f0120e1346,2026-06-23 16:51:16.110497+00:00,2026-06-23 16:30:11.416540+00:00,21,2,AND,Non-Attributed,0.53.1
1,7AD0DF66F18BEC02,2681917a-cc58-411a-aede-8798e0e6c571,2026-05-24 07:04:31.703659+00:00,2026-05-24 06:46:34.690205+00:00,17,3,AND,Non-Attributed,0.55.1
2,891384BEA345AA6E,ddabd496-0654-444d-8f09-3e9024afc3f8,2026-06-07 15:43:32.522635+00:00,2026-06-07 15:29:59.301132+00:00,13,3,AND,Non-Attributed,0.19.8
3,12FB066B6864CD4F,e8ebdaa8-5db5-4d9a-8ac4-67f265432169,2026-04-30 02:26:52.995941+00:00,2026-04-29 12:23:09.781330+00:00,843,5,AND,Non-Attributed,0.66.1
4,12FB066B6864CD4F,da9d529f-d1b2-4eef-9bc3-6078d3d65beb,2026-04-30 11:55:18.883312+00:00,2026-04-29 12:23:09.781330+00:00,1412,6,AND,Non-Attributed,0.66.1
...,...,...,...,...,...,...,...,...,...
78781,EF841C0933481632,c2e01a4c-7730-43af-8b3c-631ec56b8d1d,2026-06-28 22:29:47.532897+00:00,2026-06-28 22:02:00.597387+00:00,27,4,IOS,Non-Attributed,0.78.0
78782,EF841C0933481632,1a895e70-4629-4b7d-bed0-45c73a6bce34,2026-06-29 04:25:33.251539+00:00,2026-06-28 22:02:00.597387+00:00,383,5,IOS,Non-Attributed,0.78.0
78783,EF841C0933481632,34aedaad-c806-42ce-95a6-41b8669aa953,2026-06-29 21:14:59.765301+00:00,2026-06-28 22:02:00.597387+00:00,1392,6,IOS,Non-Attributed,0.78.0
78784,F9681C992ABAD053,d8d99635-edad-4a70-8b29-3c71c0ea2ee2,2026-06-24 03:36:43.004773+00:00,2026-06-24 03:24:45.209967+00:00,11,2,IOS,Non-Attributed,0.78.0


## Process data

In [124]:
# hide-output
# Cap every cohort's observation window to days_since_install_cap so results are comparable
# (the SQL install_cutoff_date already guarantees each included cohort could reach this many days)
data = data[data['days_since_install'] <= days_since_install_cap]

data

,user_id,dt,install_dt,country_code,days_since_install,max_level,platform,acquisition_type,install_build_version
0,396FEC7AADDE50F4,2026-04-01,2026-04-01,BR,0,6,AND,Non-Attributed,0.72.1
3,75884781C5464159,2026-04-03,2026-04-01,BR,2,5,AND,Non-Attributed,0.72.1
4,A1665AA88EC78397,2026-04-08,2026-04-01,ES,7,10,IOS,Non-Attributed,0.72.1
7,37F42D0315624B1,2026-04-06,2026-04-01,ES,5,10,IOS,Non-Attributed,0.72.1
10,C5F4665972F4E30B,2026-04-18,2026-04-01,NG,17,2,IOS,Non-Attributed,0.72.1
...,...,...,...,...,...,...,...,...,...
471691,18B42136D8F4DCF6,2026-06-30,2026-06-30,US,0,2,IOS,Non-Attributed,0.78.0
471692,49FA94331B9C272E,2026-07-03,2026-06-30,US,3,8,AND,Non-Attributed,0.77.0
471693,5FEDA941B224DB20,2026-07-21,2026-06-30,SE,21,21,AND,Non-Attributed,0.78.0
471694,E0C8CADE640DC720,2026-07-23,2026-06-30,US,23,30,AND,Influencer,0.78.0


## Player level progression

In [125]:
# hide-output
# Weighted average max level reached by days since install (drop buckets with <50 users)
min_bucket_size = 50

level_by_day = data.groupby(['days_since_install', 'max_level']).agg(
    users=('user_id', 'nunique')
).reset_index()

day_totals = level_by_day.groupby('days_since_install').agg(
    total_users=('users', 'sum')
).reset_index()

level_by_day = level_by_day.merge(day_totals, on='days_since_install')
level_by_day = level_by_day[level_by_day['users'] >= min_bucket_size]

avg_level_by_day = level_by_day.groupby('days_since_install').apply(
    lambda x: pd.Series({
        'weighted_avg_max_level': (x['max_level'] * x['users']).sum() / x['users'].sum(),
        'cohort_users': x['users'].sum(),
    }),
    include_groups=False
).reset_index()

avg_level_by_day

,days_since_install,weighted_avg_max_level,cohort_users
0,0,3.190753,52948.0
1,1,5.017430,22720.0
2,2,6.174155,16382.0
3,3,7.100870,13800.0
4,4,7.890357,12185.0
5,5,8.551476,11112.0
6,6,9.162134,10442.0
7,7,9.642533,10152.0
8,8,10.240636,9371.0
9,9,10.860605,8623.0


### Average + percentile max level reached

In [126]:
# hide-output
# P10 / P50 / P90 max level reached by days since install, weighted by user counts
def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)

level_dist = data.groupby(['days_since_install', 'max_level']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts = level_dist.groupby('days_since_install').apply(
    weighted_quantiles, include_groups=False
).reset_index()

# Total users considered at each day bucket, for hover display
day_totals_for_pcts = level_dist.groupby('days_since_install')['users'].sum().reset_index(name='total_users')
level_pcts = level_pcts.merge(day_totals_for_pcts, on='days_since_install')

level_pcts

,days_since_install,p10,p50,p90,total_users
0,0,1,3,5,53004
1,1,3,5,7,22838
2,2,3,6,10,16531
3,3,4,7,11,13943
4,4,4,8,12,12363
5,5,5,9,13,11320
6,6,5,10,14,10654
7,7,5,10,15,10364
8,8,5,11,16,9590
9,9,6,11,16,8937


In [127]:
# Chart 1 — average + P10/P50/P90 max level band chart by days since install
df = level_pcts.merge(
    avg_level_by_day[['days_since_install', 'weighted_avg_max_level']],
    on='days_since_install', how='left',
).sort_values('days_since_install')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['p10'],
    mode='lines', line=dict(width=0), showlegend=False,
    customdata=df[['total_users']],
    hovertemplate='P10: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['p90'],
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(59,130,246,0.15)',
    name='P10–P90',
    customdata=df[['total_users']],
    hovertemplate='P90: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['p50'],
    mode='lines+markers', line=dict(width=2.5, color='rgba(59,130,246,1)'),
    name='P50 (median)',
    customdata=df[['total_users']],
    hovertemplate='P50: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df['days_since_install'], y=df['weighted_avg_max_level'],
    mode='lines', line=dict(width=1.5, color='rgba(30,41,59,0.85)', dash='dash'),
    name='Mean',
    hovertemplate='Mean: %{y:.2f}<extra></extra>',
))

fig.update_layout(
    title='Player level: average + P10/P50/P90 by days since install',
    xaxis_title='Days since install',
    yaxis_title='Max level',
    width=1000, height=500,
)
fig.show()

#### Summary table

In [128]:
# Same numbers as the chart above, in table form — easier to read exact values or paste elsewhere
days_summary_table = df[['days_since_install', 'weighted_avg_max_level', 'p10', 'p50', 'p90', 'total_users']].rename(columns={
    'days_since_install': 'Days since install',
    'weighted_avg_max_level': 'Mean level',
    'p10': 'P10',
    'p50': 'P50',
    'p90': 'P90',
    'total_users': 'Users',
})
days_summary_table['Mean level'] = days_summary_table['Mean level'].round(1)
days_summary_table

,Days since install,Mean level,P10,P50,P90,Users
0,0,3.2,1,3,5,53004
1,1,5.0,3,5,7,22838
2,2,6.2,3,6,10,16531
3,3,7.1,4,7,11,13943
4,4,7.9,4,8,12,12363
5,5,8.6,5,9,13,11320
6,6,9.2,5,10,14,10654
7,7,9.6,5,10,15,10364
8,8,10.2,5,11,16,9590
9,9,10.9,6,11,16,8937


## Level funnel

In [129]:
# hide-output
# Cumulative % of users reaching each level, based on each user's overall max level in the window
funnel_level_cap = 30  # a handful of users reach much higher levels; capped for readability

user_max_level = data.groupby('user_id')['max_level'].max()
total_users = user_max_level.shape[0]

levels = list(range(1, funnel_level_cap + 1))
funnel = pd.DataFrame({'level': levels})
funnel['users_reached'] = funnel['level'].apply(lambda l: (user_max_level >= l).sum())
funnel['pct_reached'] = funnel['users_reached'] / total_users

funnel

,level,users_reached,pct_reached
0,1,53532,1.000000
1,2,47852,0.893895
2,3,41463,0.774546
3,4,30946,0.578084
4,5,23785,0.444314
5,6,18282,0.341515
6,7,13846,0.258649
7,8,12156,0.227079
8,9,10939,0.204345
9,10,10134,0.189307


In [130]:
# Chart 2 — % of users reaching each level (cumulative), vertical bar chart with user counts
fig = go.Figure(go.Bar(
    x=[f'L{l}' for l in funnel['level']],
    y=funnel['pct_reached'],
    text=funnel['users_reached'].apply(lambda u: f'{u:,}'),
    textposition='outside',
    marker_color='rgba(59,130,246,0.85)',
    hovertemplate='%{x}: %{y:.1%} reached<br>n=%{text} users<extra></extra>',
))

fig.update_layout(
    title='% of users reaching each level (cumulative), labeled with user count',
    xaxis_title='Level',
    yaxis_title='% of users reached',
    yaxis_tickformat='.0%',
    width=1200, height=550,
    uniformtext_minsize=8,
    uniformtext_mode='hide',
)
fig.show()

## Level progression by minutes since install (Day 0 only)

Uses session-level data — `fact_ssdt_user_ssid` (precise `session_start_ts` per session) joined with `fact_ssdt_user_ssid_level_progression` (max level reached per session) — to get true per-minute resolution instead of calendar-day buckets. Restricted to each user's **Day 0** (their first `minutes_since_install_cap` minutes after install) — the window where minute-level resolution actually adds insight over the daily charts above. Same install-window population and maturity guarantee as above (`start_date` → `install_cutoff_date`).

In [131]:
# hide-output
# SQL already restricts to Day 0 (minutes_since_install_cap); collapse multiple sessions
# in the same minute bucket down to each user's highest level reached in that bucket
minute_data = minute_data[minute_data['minutes_since_install'] <= minutes_since_install_cap]
minute_data = minute_data.groupby(['user_id', 'minutes_since_install'])['max_level'].max().reset_index()

minute_data

,user_id,minutes_since_install,max_level
0,1000634FDA7838D2,10,2
1,100186873D03CE09,14,3
2,100186873D03CE09,51,4
3,100186873D03CE09,58,5
4,10048987539D074A,241,3
...,...,...,...
78712,FFFC6EC3E9B50ACC,1341,4
78713,FFFD05D0A3EFE3B4,3,2
78714,FFFE079DACB6000A,33,5
78715,FFFEB27F8529987F,27,5


In [132]:
# hide-output
# Weighted average max level reached by minutes since install (drop buckets with <50 users)
minute_level_by_minute = minute_data.groupby(['minutes_since_install', 'max_level']).agg(
    users=('user_id', 'nunique')
).reset_index()

minute_level_by_minute = minute_level_by_minute[minute_level_by_minute['users'] >= min_bucket_size]

minute_avg_level_by_minute = minute_level_by_minute.groupby('minutes_since_install').apply(
    lambda x: pd.Series({
        'weighted_avg_max_level': (x['max_level'] * x['users']).sum() / x['users'].sum(),
        'cohort_users': x['users'].sum(),
    }),
    include_groups=False
).reset_index()

minute_avg_level_by_minute

,minutes_since_install,weighted_avg_max_level,cohort_users
0,2,2.000000,323.0
1,3,2.000000,654.0
2,4,2.000000,927.0
3,5,2.069703,1076.0
4,6,2.155624,1298.0
5,7,2.288376,1342.0
6,8,2.415026,1371.0
7,9,2.526316,1482.0
8,10,2.624691,1620.0
9,11,2.741012,1641.0


### Average + percentile max level reached

In [133]:
# hide-output
# P10 / P50 / P90 max level reached by minutes since install, weighted by user counts
# (reuses the weighted_quantiles function defined earlier in the notebook)
minute_level_dist = minute_data.groupby(['minutes_since_install', 'max_level']).agg(
    users=('user_id', 'count')
).reset_index()

minute_level_pcts = minute_level_dist.groupby('minutes_since_install').apply(
    weighted_quantiles, include_groups=False
).reset_index()

# Total users considered at each minute bucket, for hover display
minute_totals = minute_level_dist.groupby('minutes_since_install')['users'].sum().reset_index(name='total_users')
minute_level_pcts = minute_level_pcts.merge(minute_totals, on='minutes_since_install')

minute_level_pcts

,minutes_since_install,p10,p50,p90,total_users
0,1,2,2,2,28
1,2,2,2,2,323
2,3,2,2,2,656
3,4,2,2,2,944
4,5,2,2,2,1076
...,...,...,...,...,...
1435,1436,3,5,7,27
1436,1437,4,6,8,31
1437,1438,3,5,8,28
1438,1439,4,5,7,40


In [137]:
# Chart 3 — average + P10/P50/P90 max level band chart by minutes since install (Day 0)
df_minute = minute_level_pcts.merge(
    minute_avg_level_by_minute[['minutes_since_install', 'weighted_avg_max_level']],
    on='minutes_since_install', how='left',
)
df_minute = df_minute[df_minute.minutes_since_install <= 60].sort_values('minutes_since_install')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['p10'],
    mode='lines', line=dict(width=0), showlegend=False,
    customdata=df_minute[['total_users']],
    hovertemplate='P10: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['p90'],
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(239,68,68,0.15)',
    name='P10–P90',
    customdata=df_minute[['total_users']],
    hovertemplate='P90: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['p50'],
    mode='lines', line=dict(width=2.5, color='rgba(239,68,68,1)'),
    name='P50 (median)',
    customdata=df_minute[['total_users']],
    hovertemplate='P50: %{y:.0f}<br>n=%{customdata[0]:,} users<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df_minute['minutes_since_install'], y=df_minute['weighted_avg_max_level'],
    mode='lines', line=dict(width=1.5, color='rgba(30,41,59,0.85)', dash='dash'),
    name='Mean',
    hovertemplate='Mean: %{y:.2f}<br>Minutes since install: %{x}<extra></extra>',
))

fig.update_layout(
    title='Player level: average + P10/P50/P90 by minutes since install (Day 0)',
    xaxis_title='Minutes since install',
    yaxis_title='Max level',
    width=1000, height=500,
)
fig.show()

#### Summary table

In [135]:
# Same numbers as the chart above, in table form — easier to read exact values or paste elsewhere
minute_summary_table = df_minute[['minutes_since_install', 'weighted_avg_max_level', 'p10', 'p50', 'p90', 'total_users']].rename(columns={
    'minutes_since_install': 'Minutes since install',
    'weighted_avg_max_level': 'Mean level',
    'p10': 'P10',
    'p50': 'P50',
    'p90': 'P90',
    'total_users': 'Users',
})
minute_summary_table['Mean level'] = minute_summary_table['Mean level'].round(1)
minute_summary_table

,Minutes since install,Mean level,P10,P50,P90,Users
0,1,NaN,2,2,2,28
1,2,2.0,2,2,2,323
2,3,2.0,2,2,2,656
3,4,2.0,2,2,2,944
4,5,2.1,2,2,2,1076
5,6,2.2,2,2,3,1299
6,7,2.3,2,2,3,1342
7,8,2.4,2,2,3,1378
8,9,2.5,2,3,3,1495
9,10,2.6,2,3,3,1639


In [136]:
# hide-output
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./level_progression_simple.ipynb',
    output_path='./level_progression_simple.html',
)

Saved to level_progression_simple.html


PosixPath('level_progression_simple.html')